In [ ]:
import os
import json
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

In [ ]:
dir_data = "/mnt/c/Users/mbrudfors/Data/Learn2Reg/OASIS/" # Download from: https://cloud.imi.uni-luebeck.de/s/xcZrLSQYtK68em8
json_meta = os.path.join(dir_data, "OASIS_dataset.json")
dir_out = "OASIS_registered"
os.makedirs(dir_out, exist_ok=True)

In [ ]:
with open(json_meta, 'r') as f:
    meta = json.load(f)

In [ ]:
def get_pair(meta, i, dir_data, verbose=False):
    if i >= len(meta["registration_val"]):
        raise ValueError(f"Index {i} is out of range for meta['registration_val'] of length {len(meta['registration_val'])}")
    fixed_image = os.path.normpath(os.path.join(dir_data, meta["registration_val"][i]["fixed"]))
    moving_image = os.path.normpath(os.path.join(dir_data, meta["registration_val"][i]["moving"]))
    fixed_label = fixed_image.replace("imagesTr", "labelsTr")
    moving_label = moving_image.replace("imagesTr", "labelsTr")
    pair = {
        "fixed": {"image": fixed_image, "label": fixed_label},
        "moving": {"image": moving_image, "label": moving_label},
    }
    if verbose:
        print("Fixed image: ", pair["fixed"]["image"])
        print("Fixed label: ", pair["fixed"]["label"])
        print("Moving image:", pair["moving"]["image"])
        print("Moving label:", pair["moving"]["label"])
    return pair

pair = get_pair(meta, 0, dir_data, verbose=True)

In [ ]:
def visualize_pair(fixed_image, moving_image, fixed_label=None, moving_label=None, 
                   slice_idx=None, axis=2, title="Fixed vs Moving", alpha=0.4,
                   n_contours=8, figsize=(18, 6)):
    """
    Visualize fixed and moving image/label pairs side by side with overlay.
    
    Parameters
    ----------
    fixed_image : str or np.ndarray
        Path to fixed image NIfTI file or numpy array
    moving_image : str or np.ndarray
        Path to moving image NIfTI file or numpy array
    fixed_label : str or np.ndarray, optional
        Path to fixed label NIfTI file or numpy array
    moving_label : str or np.ndarray, optional
        Path to moving label NIfTI file or numpy array
    slice_idx : int, optional
        Slice index to display. If None, uses middle slice.
    axis : int
        Axis along which to take the slice (0, 1, or 2). Default is 2 (axial).
    title : str
        Title for the figure
    alpha : float
        Transparency for label overlay (0-1)
    n_contours : int
        Number of contour levels for the overlay
    figsize : tuple
        Figure size
    
    Returns
    -------
    fig : matplotlib.figure.Figure
        The figure object
    """
    # Load images if paths are provided
    def load_vol(x):
        if isinstance(x, str):
            return nib.load(x).get_fdata()
        return x
    
    fix_img = load_vol(fixed_image)
    mov_img = load_vol(moving_image)
    fix_lbl = load_vol(fixed_label) if fixed_label is not None else None
    mov_lbl = load_vol(moving_label) if moving_label is not None else None
    
    # Determine slice index
    if slice_idx is None:
        slice_idx = fix_img.shape[axis] // 2
    
    # Extract slices
    def get_slice(vol, idx, ax):
        slc = [slice(None)] * vol.ndim
        slc[ax] = idx
        return vol[tuple(slc)]
    
    fix_slice = get_slice(fix_img, slice_idx, axis)
    mov_slice = get_slice(mov_img, slice_idx, axis)
    fix_lbl_slice = get_slice(fix_lbl, slice_idx, axis) if fix_lbl is not None else None
    mov_lbl_slice = get_slice(mov_lbl, slice_idx, axis) if mov_lbl is not None else None
    
    # Create colormap for labels (random colors, transparent background)
    label_cmap = None
    n_labels = 0
    if fix_lbl_slice is not None or mov_lbl_slice is not None:
        all_labels = []
        if fix_lbl_slice is not None:
            all_labels.extend(np.unique(fix_lbl_slice))
        if mov_lbl_slice is not None:
            all_labels.extend(np.unique(mov_lbl_slice))
        n_labels = int(max(all_labels)) + 1
        np.random.seed(42)  # For consistent colors
        colors = np.random.rand(n_labels, 4)
        colors[:, 3] = 1.0  # Full opacity
        colors[0] = [0, 0, 0, 0]  # Background transparent
        label_cmap = ListedColormap(colors)
    
    # Determine number of rows
    n_rows = 2 if (fix_lbl_slice is not None or mov_lbl_slice is not None) else 1
    n_cols = 3  # Fixed, Moving, Overlay
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    # Plot images
    axes[0, 0].imshow(fix_slice.T, cmap='gray', origin='lower', aspect='equal')
    axes[0, 0].set_title(f'Fixed Image (slice {slice_idx})')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(mov_slice.T, cmap='gray', origin='lower', aspect='equal')
    axes[0, 1].set_title(f'Moving Image (slice {slice_idx})')
    axes[0, 1].axis('off')
    
    # Overlay: Fixed image with moving contours
    axes[0, 2].imshow(fix_slice.T, cmap='gray', origin='lower', aspect='equal')
    # Compute contour levels based on image intensity range
    mov_min, mov_max = mov_slice.min(), mov_slice.max()
    levels = np.linspace(mov_min + 0.1*(mov_max-mov_min), mov_max - 0.1*(mov_max-mov_min), n_contours)
    axes[0, 2].contour(mov_slice.T, levels=levels, colors='red', linewidths=0.8, 
                       origin='lower', alpha=0.7)
    axes[0, 2].set_title('Fixed + Moving Contours')
    axes[0, 2].axis('off')
    
    # Plot labels if provided
    if n_rows > 1:
        # Fixed with label overlay
        axes[1, 0].imshow(fix_slice.T, cmap='gray', origin='lower', aspect='equal')
        if fix_lbl_slice is not None:
            axes[1, 0].imshow(fix_lbl_slice.T, cmap=label_cmap, origin='lower', 
                             aspect='equal', alpha=alpha, vmin=0, vmax=n_labels-1)
        axes[1, 0].set_title('Fixed + Labels')
        axes[1, 0].axis('off')
        
        # Moving with label overlay
        axes[1, 1].imshow(fix_slice.T, cmap='gray', origin='lower', aspect='equal')
        if mov_lbl_slice is not None:
            axes[1, 1].imshow(mov_lbl_slice.T, cmap=label_cmap, origin='lower', 
                             aspect='equal', alpha=alpha, vmin=0, vmax=n_labels-1)
        axes[1, 1].set_title('Moving Labels on Fixed')
        axes[1, 1].axis('off')
        
        # Overlay: Fixed with both label contours
        axes[1, 2].imshow(fix_slice.T, cmap='gray', origin='lower', aspect='equal')
        if fix_lbl_slice is not None:
            for lbl in np.unique(fix_lbl_slice):
                if lbl == 0:
                    continue
                axes[1, 2].contour(fix_lbl_slice.T == lbl, levels=[0.5], colors='lime', 
                                  linewidths=0.8, origin='lower')
        if mov_lbl_slice is not None:
            for lbl in np.unique(mov_lbl_slice):
                if lbl == 0:
                    continue
                axes[1, 2].contour(mov_lbl_slice.T == lbl, levels=[0.5], colors='magenta', 
                                  linewidths=0.8, origin='lower', linestyles='dashed')
        axes[1, 2].set_title('Label Contours (green=fixed, magenta=moving)')
        axes[1, 2].axis('off')
    
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    return fig


# Visualize the pair before registration
visualize_pair(
    fixed_image=pair["fixed"]["image"],
    moving_image=pair["moving"]["image"],
    fixed_label=pair["fixed"]["label"],
    moving_label=pair["moving"]["label"],
    title="Before Registration",
    figsize=(10, 8)
);

In [ ]:
def compute_dice_scores(label1, label2, exclude_zero=True):
    """
    Compute Dice score for each label between two segmentation masks.
    
    Parameters
    ----------
    label1 : str or np.ndarray
        Path to first label NIfTI file or numpy array (e.g., fixed labels)
    label2 : str or np.ndarray
        Path to second label NIfTI file or numpy array (e.g., moving labels)
    exclude_zero : bool
        Whether to exclude background (label 0) from computation
    
    Returns
    -------
    dice_scores : dict
        Dictionary mapping label ID to Dice score
    mean_dice : float
        Mean Dice score across all labels
    """
    # Load labels if paths are provided
    def load_vol(x):
        if isinstance(x, str):
            return nib.load(x).get_fdata()
        return x
    
    lbl1 = load_vol(label1)
    lbl2 = load_vol(label2)
    
    # Get all unique labels
    all_labels = np.union1d(np.unique(lbl1), np.unique(lbl2))
    if exclude_zero:
        all_labels = all_labels[all_labels != 0]
    
    dice_scores = {}
    for lbl in all_labels:
        mask1 = (lbl1 == lbl)
        mask2 = (lbl2 == lbl)
        intersection = np.sum(mask1 & mask2)
        union = np.sum(mask1) + np.sum(mask2)
        if union == 0:
            dice = 1.0 if intersection == 0 else 0.0
        else:
            dice = 2 * intersection / union
        dice_scores[int(lbl)] = dice
    
    mean_dice = np.mean(list(dice_scores.values())) if dice_scores else 0.0
    
    return dice_scores, mean_dice


def visualize_dice_scores(dice_scores, mean_dice=None, title="Dice Scores per Label", 
                          figsize=(10, 8), color_by_value=True):
    """
    Visualize Dice scores as a horizontal bar plot sorted by score.
    
    Parameters
    ----------
    dice_scores : dict
        Dictionary mapping label ID to Dice score
    mean_dice : float, optional
        Mean Dice score (computed if not provided)
    title : str
        Title for the plot
    figsize : tuple
        Figure size
    color_by_value : bool
        If True, color bars by Dice value (red=low, green=high)
    
    Returns
    -------
    fig : matplotlib.figure.Figure
        The figure object
    """
    if mean_dice is None:
        mean_dice = np.mean(list(dice_scores.values()))
    
    # Sort by Dice score
    sorted_items = sorted(dice_scores.items(), key=lambda x: x[1])
    labels = [f"Label {k}" for k, v in sorted_items]
    scores = [v for k, v in sorted_items]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Color by value: red (low) -> yellow (mid) -> green (high)
    if color_by_value:
        colors = plt.cm.RdYlGn(np.array(scores))
    else:
        colors = 'steelblue'
    
    y_pos = np.arange(len(labels))
    bars = ax.barh(y_pos, scores, color=colors, edgecolor='gray', linewidth=0.5)
    
    # Add mean line
    ax.axvline(x=mean_dice, color='navy', linestyle='--', linewidth=2, 
               label=f'Mean Dice: {mean_dice:.3f}')
    
    # Formatting
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('Dice Score', fontsize=12)
    ax.set_xlim(0, 1)
    ax.set_title(f'{title}\nMean Dice: {mean_dice:.3f}', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right')
    
    # Add grid for readability
    ax.grid(axis='x', alpha=0.3)
    ax.set_axisbelow(True)
    
    # Add value labels on bars
    for i, (bar, score) in enumerate(zip(bars, scores)):
        ax.text(score + 0.01, bar.get_y() + bar.get_height()/2, 
                f'{score:.2f}', va='center', fontsize=7)
    
    plt.tight_layout()
    
    return fig


# Compute and visualize Dice scores before registration
dice_scores, mean_dice = compute_dice_scores(
    pair["fixed"]["label"], 
    pair["moving"]["label"]
)
print(f"Mean Dice Score: {mean_dice:.4f}")
visualize_dice_scores(dice_scores, mean_dice, title="Dice Scores Before Registration");

In [ ]:
# STEP 1: Nitorch image-based AFFINE registration using CLI
pth_fixed = pair["fixed"]["image"]
pth_moving = pair["moving"]["image"]
pth_affine = os.path.join(dir_out, 'affine.lta')
cmd_affine = lambda pth_fixed, pth_moving, dir_out: \
    "nitorch register" \
    + " --verbose 1" \
    + " --gpu 0" \
    + " --output-dir " + dir_out \
    + " @loss lcc" \
    + " @@fix " + pth_fixed + " --label --fwhm 1 -b dct2" \
    + " @@mov " + pth_moving + " --label --fwhm 1 -b dct2" \
    + " @affine affine @@optim -t 1e-6 -n 256" \
    + " @optim i -t 1e-6 -n 256" \
    + " @pyramid --levels 0 1 2"
cmd = cmd_affine(pth_fixed, pth_moving, dir_out)
# print(">> " + cmd)
os.system(cmd)

In [ ]:
# STEP 1: Nitorch label-based AFFINE registration using CLI
pth_fixed = pair["fixed"]["label"]
pth_moving = pair["moving"]["label"]
pth_affine = os.path.join(dir_out, 'affine.lta')
cmd_affine = lambda pth_fixed, pth_moving, dir_out: \
    "nitorch register" \
    + " --verbose 1" \
    + " --gpu 0" \
    + " --output-dir " + dir_out \
    + " @loss dice" \
    + " @@fix " + pth_fixed + " --label --fwhm 1 -b dct2" \
    + " @@mov " + pth_moving + " --label --fwhm 1 -b dct2" \
    + " @affine affine @@optim -t 1e-6 -n 256" \
    + " @optim i -t 1e-6 -n 256" \
    + " @pyramid --levels 0 1 2"
cmd = cmd_affine(pth_fixed, pth_moving, dir_out)
# print(">> " + cmd)
os.system(cmd)
